In [1]:
import torch
import numpy as np

import os
import sys
import numpy
import random
import statistics
import open3d as o3d

sys.path.append(os.path.abspath(".."))

from models.vae_adain import Model as VAE
from default_config import cfg

cfg_original = cfg.clone()
cfg_mirrored = cfg.clone()

sys.path.append(os.path.abspath("../.."))

from data.Datasets.Scripts.Symmetry_Computation.Householder_transform import householder_transformation

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Using /home/ncaytuir/.cache/torch_extensions/py38_cu111 as PyTorch extensions root...
Detected CUDA files, patching ldflags
Emitting ninja build file /home/ncaytuir/.cache/torch_extensions/py38_cu111/emd_ext/build.ninja...
Building extension module emd_ext...
Allowing ninja to set a default number of workers... (overridable by setting the environment variable MAX_JOBS=N)
ninja: no work to do.
Loading extension module emd_ext...
load emd_ext time: 0.179s
utils/utils.py: USE_COMET=1, USE_WB=0


In [2]:
# Path to ckpt
ckpt_path_original = "/home/ncaytuir/LION_necs/lion_ckpt/unconditional/airplane/checkpoints/vae_only.pt"
cfg_path_original = "/home/ncaytuir/LION_necs/lion_ckpt/unconditional/airplane/cfg.yml"

ckpt_path_mirrored = "/home/ncaytuir/LION_necs/lion_ckpt/unconditional_new_dataset/airplane/epoch_7999_iters_175999.pt"
cfg_path_mirrored = "/home/ncaytuir/LION_necs/lion_ckpt/unconditional_new_dataset/airplane/cfg.yml"

input_original_dir = "/home/ncaytuir/data/Datasets/ShapeNetCore.v2.PC15k/02691156/val"
input_mirrored_dir = "/home/ncaytuir/data/Datasets/Mirrored_ShapeNetCore.v3.PC15k/02691156/val"

shape1 = "/home/ncaytuir/data/Datasets/ShapeNetCore.v2.PC15k/02691156/val/1a54a2319e87bd4071d03b466c72ce41.npy"

cfg_original.merge_from_file(cfg_path_original)
args_original = cfg_original

cfg_mirrored.merge_from_file(cfg_path_mirrored)
args_mirrored = cfg_mirrored

# Load ckpt
ckpt_original = torch.load(ckpt_path_original, map_location=device)
ckpt_mirrored = torch.load(ckpt_path_mirrored, map_location=device)

# Instanciate model
vae_original = VAE(args_original).to(device)
vae_original.load_state_dict(ckpt_original["model"])
vae_original.eval()

vae_mirrored = VAE(args_mirrored).to(device)
vae_mirrored.load_state_dict(ckpt_mirrored["model"])
vae_mirrored.eval()

#print("args", args)

2026-04-02 17:45:12.518 | INFO     | utils.model_helper:import_model:114 - import: models.shapelatent_modules.PointNetPlusEncoder


Using /home/ncaytuir/.cache/torch_extensions/py38_cu111 as PyTorch extensions root...
Detected CUDA files, patching ldflags
Emitting ninja build file /home/ncaytuir/.cache/torch_extensions/py38_cu111/_pvcnn_backend/build.ninja...
Building extension module _pvcnn_backend...
Allowing ninja to set a default number of workers... (overridable by setting the environment variable MAX_JOBS=N)


2026-04-02 17:45:12.801 | INFO     | models.shapelatent_modules:__init__:29 - [Encoder] zdim=128, out_sigma=True; force_att: 0
2026-04-02 17:45:12.802 | INFO     | utils.model_helper:import_model:114 - import: models.latent_points_ada.PointTransPVC
2026-04-02 17:45:12.804 | INFO     | models.latent_points_ada:__init__:38 - [Build Unet] extra_feature_channels=0, input_dim=3
2026-04-02 17:45:12.877 | INFO     | utils.model_helper:import_model:114 - import: models.latent_points_ada.LatentPointDecPVC
2026-04-02 17:45:12.878 | INFO     | models.latent_points_ada:__init__:241 - [Build Dec] point_dim=3, context_dim=1
2026-04-02 17:45:12.878 | INFO     | models.latent_points_ada:__init__:38 - [Build Unet] extra_feature_channels=1, input_dim=3
2026-04-02 17:45:12.948 | INFO     | models.vae_adain:__init__:50 - [Build Model] style_encoder: models.shapelatent_modules.PointNetPlusEncoder, encoder: models.latent_points_ada.PointTransPVC, decoder: models.latent_points_ada.LatentPointDecPVC


ninja: no work to do.
Loading extension module _pvcnn_backend...


2026-04-02 17:45:13.068 | INFO     | utils.model_helper:import_model:114 - import: models.shapelatent_modules.PointNetPlusEncoder
2026-04-02 17:45:13.071 | INFO     | models.shapelatent_modules:__init__:29 - [Encoder] zdim=128, out_sigma=True; force_att: 0
2026-04-02 17:45:13.072 | INFO     | utils.model_helper:import_model:114 - import: models.latent_points_ada.PointTransPVC
2026-04-02 17:45:13.072 | INFO     | models.latent_points_ada:__init__:38 - [Build Unet] extra_feature_channels=0, input_dim=3
2026-04-02 17:45:13.130 | INFO     | utils.model_helper:import_model:114 - import: models.latent_points_ada.LatentPointDecPVC
2026-04-02 17:45:13.131 | INFO     | models.latent_points_ada:__init__:241 - [Build Dec] point_dim=3, context_dim=1
2026-04-02 17:45:13.131 | INFO     | models.latent_points_ada:__init__:38 - [Build Unet] extra_feature_channels=1, input_dim=3
2026-04-02 17:45:13.193 | INFO     | models.vae_adain:__init__:50 - [Build Model] style_encoder: models.shapelatent_modules.P

Model(
  (style_encoder): PointNetPlusEncoder(
    (mlp): Linear(in_features=64, out_features=256, bias=True)
    (layers): ModuleList(
      (0): Sequential(
        (0): PVConv(
          (voxelization): Voxelization(resolution=32, normalized eps = 0)
          (voxel_layers): Sequential(
            (0): Conv3d(3, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
            (1): GroupNorm(8, 32, eps=1e-05, affine=True)
            (2): Swish()
            (3): Dropout(p=0.1, inplace=False)
            (4): Conv3d(32, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
            (5): GroupNorm(8, 32, eps=1e-05, affine=True)
            (6): SE(32, 32)
          )
          (point_features): SharedMLP(
            (layers): Sequential(
              (0): Conv1d(3, 32, kernel_size=(1,), stride=(1,))
              (1): GroupNorm(8, 32, eps=1e-05, affine=True)
              (2): Swish()
            )
          )
        )
        (1): PVConv(
          (voxel

### Helpers

In [3]:
# Mean and std for original objects
## Airplane
original_airplane_mean = torch.tensor([0.0004, 0.0061, 0.0488], device=device).view(1, 1, 3) # [1, 1, 3]
original_airplane_std = torch.tensor([0.1201], device=device).view(1, 1, 1) # [1, 1, 1]

# Mean and std for mirrored objects
## Airplane
mirrored_airplane_mean = torch.tensor([0.0002, 0.0066, 0.0589], device=device).view(1, 1, 3) # [1, 1, 3]
mirrored_airplane_std = torch.tensor([0.1171], device=device).view(1, 1, 1) # [1, 1, 1]

In [4]:
def compute_average_l2_from_pairs(files, directory, vae):
    l2 = []

    for a in files:
        file_path_A = os.path.join(directory, a)

        # Take another random pc
        b = random.choice(files)

        if b == a:
            b = random.choice(files)

        #print("A:", a)
        #print("B:", b)
        
        file_path_B = os.path.join(directory, b)

        # Load pcs
        pc_A = np.load(file_path_A)
        pc_B = np.load(file_path_B)

        if pc_A.ndim == 2 and pc_B.ndim == 2: # Matriz
            pc_A = pc_A[None, ...]
            pc_B = pc_B[None, ...]

        A_tensor = torch.tensor(pc_A, dtype=torch.float32).to(device)
        B_tensor = torch.tensor(pc_B, dtype=torch.float32).to(device)

        # Get latents (mu)
        if vae == "vae_original":
            A_tensor = (A_tensor - original_airplane_mean) / original_airplane_std
            B_tensor = (B_tensor - original_airplane_mean) / original_airplane_std
            
            mu_A = vae_original.get_latents_mu(A_tensor)
            mu_B = vae_original.get_latents_mu(B_tensor)

        elif vae == "vae_mirrored":
            A_tensor = (A_tensor - mirrored_airplane_mean) / mirrored_airplane_std
            B_tensor = (B_tensor - mirrored_airplane_mean) / mirrored_airplane_std

            mu_A = vae_mirrored.get_latents_mu(A_tensor)
            mu_B = vae_mirrored.get_latents_mu(B_tensor)

        # Compute l2
        computed_l2 = torch.norm(mu_A[0] - mu_B[0], dim=-1)

        l2.append(computed_l2.item())

    print("Mean:", statistics.mean(l2))
    print("Max:", max(l2))
    print("Min:", min(l2))

    return statistics.mean(l2)

def compute_l2_list_from_original_mirrored(files, directory, average_l2_from_pairs, vae):
    means = []
    cont = 0

    for x in files:
        file_path_pc = os.path.join(directory, x)
        original_pc = np.load(file_path_pc)

        if original_pc.shape[0] == 15000:
            pass
        elif original_pc.shape[0] == 30000:
            #pass
            original_pc = farthest_point_sampling(original_pc, 15000)
        
        # Obtain the mirrored version
        mirrored_pc = householder_transformation(original_pc)

        
        #np.save(f"latent_results/original_pc_{cont}", original_pc)
        #np.save(f"latent_results/mirrored_pc_{cont}", mirrored_pc)

        """ if cont == 3:
            break """

        # Convert to tensors
        if original_pc.ndim == 2 and mirrored_pc.ndim == 2:
            original_pc = original_pc[None, ...]
            mirrored_pc = mirrored_pc[None, ...]
        
        original_pc = torch.tensor(original_pc, dtype=torch.float32).to(device)
        mirrored_pc = torch.tensor(mirrored_pc, dtype=torch.float32).to(device)
    
        if vae == "vae_original":
            original_pc = (original_pc - original_airplane_mean) / original_airplane_std
            mirrored_pc = (mirrored_pc - original_airplane_mean) / original_airplane_std

            mu_original = vae_original.get_latents_mu(original_pc)
            mu_mirrored = vae_original.get_latents_mu(mirrored_pc)

        elif vae == "vae_mirrored":
            original_pc = (original_pc - mirrored_airplane_mean) / mirrored_airplane_std
            mirrored_pc = (mirrored_pc - mirrored_airplane_mean) / mirrored_airplane_std

            mu_original = vae_mirrored.get_latents_mu(original_pc)
            mu_mirrored = vae_mirrored.get_latents_mu(mirrored_pc)

        numerator = torch.norm(mu_original[0] - mu_mirrored[0], dim=-1)
        denominator = average_l2_from_pairs

        result = numerator / denominator

        print("Numerator:", numerator.item())
        print("Denominator:", denominator)
        print("Result:", result.item())

        means.append(result.item())

        cont += 1

    return means

def farthest_point_sampling(points, n_samples=2048):
    # Create a point cloud of Open3D
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points)

    # compute FPS
    downpcd_farthest = pcd.farthest_point_down_sample(n_samples)

    return np.asarray(downpcd_farthest.points)

## Latents from original encoder

In [5]:
original_files = sorted([f for f in os.listdir(input_original_dir) if f.endswith(".npy")])#[:5]

average_l2_from_pairs = compute_average_l2_from_pairs(original_files[:100], input_original_dir, "vae_original")
l2_list_from_original_mirrored = compute_l2_list_from_original_mirrored(original_files, input_original_dir, average_l2_from_pairs, "vae_original")

print(l2_list_from_original_mirrored)
print("\nAverage mean " + str(len(original_files)) +":", statistics.mean(l2_list_from_original_mirrored))
print("Average mean/max:", max(l2_list_from_original_mirrored))
print("Average mean/min:", min(l2_list_from_original_mirrored))

Mean: 2.0111634504795073
Max: 3.648311138153076
Min: 0.7806313633918762
Numerator: 0.32410651445388794
Denominator: 2.0111634504795073
Result: 0.16115374863147736
Numerator: 0.3839947581291199
Denominator: 2.0111634504795073
Result: 0.19093164801597595
Numerator: 0.08072726428508759
Denominator: 2.0111634504795073
Result: 0.040139585733413696
Numerator: 0.18037274479866028
Denominator: 2.0111634504795073
Result: 0.08968576788902283
Numerator: 0.34296658635139465
Denominator: 2.0111634504795073
Result: 0.17053143680095673
Numerator: 0.3569069504737854
Denominator: 2.0111634504795073
Result: 0.17746292054653168
Numerator: 0.1860552430152893
Denominator: 2.0111634504795073
Result: 0.09251125156879425
Numerator: 0.24718010425567627
Denominator: 2.0111634504795073
Result: 0.12290403246879578
Numerator: 0.18619918823242188
Denominator: 2.0111634504795073
Result: 0.0925828218460083
Numerator: 0.455873966217041
Denominator: 2.0111634504795073
Result: 0.22667177021503448
Numerator: 0.1504472494

## Latents from mirrored encoder

In [6]:
mirrored_files = sorted([f for f in os.listdir(input_mirrored_dir) if f.endswith(".npy")])#[:5]

average_l2_from_pairs = compute_average_l2_from_pairs(mirrored_files[:100], input_mirrored_dir, "vae_mirrored")
l2_list_from_original_mirrored = compute_l2_list_from_original_mirrored(mirrored_files, input_mirrored_dir, average_l2_from_pairs, "vae_mirrored")

print(l2_list_from_original_mirrored)
print("\nAverage mean " + str(len(mirrored_files)) +":", statistics.mean(l2_list_from_original_mirrored))
print("Average mean/max:", max(l2_list_from_original_mirrored))
print("Average mean/min:", min(l2_list_from_original_mirrored))

Mean: 2.135882420539856
Max: 4.9443888664245605
Min: 0.7842274904251099
Numerator: 0.13231900334358215
Denominator: 2.135882420539856
Result: 0.06195051223039627
Numerator: 0.19306126236915588
Denominator: 2.135882420539856
Result: 0.09038946777582169
Numerator: 0.07164253294467926
Denominator: 2.135882420539856
Result: 0.03354235738515854
Numerator: 0.1804320067167282
Denominator: 2.135882420539856
Result: 0.08447656780481339
Numerator: 0.1304854452610016
Denominator: 2.135882420539856
Result: 0.06109205633401871
Numerator: 0.08486910164356232
Denominator: 2.135882420539856
Result: 0.039734914898872375
Numerator: 0.09227996319532394
Denominator: 2.135882420539856
Result: 0.04320460930466652
Numerator: 0.11782566457986832
Denominator: 2.135882420539856
Result: 0.05516486614942551
Numerator: 0.15148180723190308
Denominator: 2.135882420539856
Result: 0.0709223523736
Numerator: 0.14827856421470642
Denominator: 2.135882420539856
Result: 0.06942262500524521
Numerator: 0.1005418449640274
Den